# NB-501: LoRA Fine-tuning

Implement LoRA (Low-Rank Adaptation) for efficient fine-tuning.

## Learning Objectives
- Understand LoRA architecture
- Apply LoRA to transformer models
- Fine-tune with LoRA
- Compare with full fine-tuning

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

print("Libraries imported")

## 1. Load Base Model

In [ ]:
# TODO: Load model and tokenizer
model_name = "gpt2"  # Use smaller model for demo

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Configure LoRA

In [ ]:
# TODO: Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # Rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["c_attn"],  # Apply to attention layers
    bias="none",
)

print(f"LoRA config: r={lora_config.r}, alpha={lora_config.lora_alpha}")

## 3. Apply LoRA to Model

In [ ]:
# TODO: Apply LoRA
peft_model = get_peft_model(model, lora_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

print(f"\nTotal parameters: {sum(p.numel() for p in peft_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,}")

## 4. Prepare Training Data

In [ ]:
# TODO: Create sample dataset
train_texts = [
    "The capital of France is Paris.",
    "Machine learning is a subset of AI.",
    "Python is a programming language.",
    "The Earth revolves around the Sun.",
] * 50  # Repeat for more data

# Tokenize
encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=32,
    return_tensors="pt"
)

print(f"Training samples: {len(train_texts)}")

## 5. Training Loop

In [ ]:
# TODO: Setup training
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    logging_steps=100,
    learning_rate=1e-4,
    weight_decay=0.01,
)

# Create simple dataset class
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.encodings["input_ids"][idx].clone(),
        }

train_dataset = SimpleDataset(encodings)

print("Training setup complete")

## 6. Generate with Fine-tuned Model

In [ ]:
# TODO: Generate text
prompt = "The capital of"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate with base model
with torch.no_grad():
    base_output = model.generate(**inputs, max_new_tokens=10)

# Generate with LoRA model
with torch.no_grad():
    lora_output = peft_model.generate(**inputs, max_new_tokens=10)

print(f"Base model: {tokenizer.decode(base_output[0])}")
print(f"LoRA model: {tokenizer.decode(lora_output[0])}")

## Exercise

1. Try different ranks (r=4, 16, 32)
2. Apply LoRA to different modules
3. Compare training time vs full fine-tuning